[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/heitorramos/icd/blob/main/exemplos/25-knn-pratica-ml/notebook-colab.ipynb)


In [ ]:
# Preparação automática para execução no Google Colab.
# Fora do Colab, esta célula não altera o diretório de trabalho.
try:
    import google.colab  # type: ignore
except ImportError:
    pass
else:
    import os
    import subprocess
    from pathlib import Path

    repository = Path("/content/icd")
    if not repository.exists():
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/heitorramos/icd.git", str(repository)
        ], check=True)
    os.chdir(repository / "exemplos/25-knn-pratica-ml")
    print("Material preparado em:", Path.cwd())


# Aula 25 — KNN e a prática de machine learning

Guia de estudo executável

## Como estudar este capítulo

Este capítulo usa o KNN para percorrer uma prática completa de machine
learning. O método é simples: para classificar uma nova observação,
procuramos exemplos de treinamento próximos e usamos suas classes como
evidência. A simplicidade ajuda a enxergar decisões que também aparecem
em métodos mais sofisticados.

O KNN praticamente não possui uma etapa de treinamento tradicional; o
trabalho ocorre na previsão. Por isso, escala, escolha da distância,
número de vizinhos e presença de atributos irrelevantes têm grande
impacto. A validação cruzada ajuda a escolher $k$, mas não substitui a
análise do problema nem autoriza interpretação clínica.

Ao estudar, siga o percurso da observação nova: ela passa pelo mesmo
tratamento de valores ausentes e pela mesma padronização aprendida no
treino; suas distâncias são calculadas; os vizinhos são selecionados; os
votos são agregados; e a previsão é comparada com o resultado observado.
As métricas resumem diferentes tipos de acerto e erro e precisam ser
lidas à luz do objetivo.

## 1. Pergunta e base de dados

Este material acompanha uma análise completa: queremos classificar
perfis associados à doença renal crônica (DRC). A base tem 400
pacientes, 24 atributos preditores e a resposta `classification`. Há
medidas numéricas (creatinina, hemoglobina, glicose) e indicadores
categóricos (hipertensão, diabetes, anemia), além de valores ausentes.

> **Escopo**
>
> O estudo é didático. Desempenho nesta amostra não autoriza uso
> clínico: seriam necessárias validação externa, avaliação de vieses,
> governança e supervisão profissional.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style="whitegrid", context="notebook")
DATA = Path("data/kidney_disease.csv")
df = pd.read_csv(DATA)
df.shape

## 2. Limpeza e descrição

In [ ]:
for col in df.select_dtypes(include=["object", "string"]):
    df[col] = df[col].astype("string").str.strip().replace({"?": pd.NA, "": pd.NA})
df["classification"] = df["classification"].str.replace("\t", "", regex=False).str.strip()

numeric = ["age","bp","sg","al","su","bgr","bu","sc","sod","pot","hemo","pcv","wc","rc"]
categorical = ["rbc","pc","pcc","ba","htn","dm","cad","appet","pe","ane"]
for col in numeric:
    df[col] = pd.to_numeric(df[col], errors="coerce")
for col in categorical:
    df[col] = df[col].astype(object).where(df[col].notna(), np.nan)

df["classification"].value_counts()

In [ ]:
summary = df[numeric].describe().T[["count", "mean", "std", "min", "50%", "max"]]
summary.round(2)

In [ ]:
missing = (100 * df[numeric + categorical].isna().mean()).sort_values(ascending=False)
missing.head(12).round(1).to_frame("% ausente")

> **Interpretação**
>
> A classe DRC é majoritária e a ausência de dados não é rara. Excluir
> todas as linhas incompletas reduziria a amostra e poderia alterar sua
> composição. Por isso, a imputação fará parte da pipeline e será
> aprendida somente no treinamento.

## 3. Definição matemática do KNN

Temos $\mathcal D=\{(x_i,y_i)\}_{i=1}^{n}$, com $x_i\in\mathbb R^p$ e
$y_i\in\{0,1\}$. A distância euclidiana é

$$d_2(x,z)=\sqrt{\sum_{j=1}^{p}(x_j-z_j)^2}.$$

Aqui, $p$ é o número de atributos; $x_j$ e $z_j$ são os valores na
coordenada $j$. Se $N_k(x)$ contém os índices dos $k$ exemplos mais
próximos de $x$, então

$$\hat p_k(x)=\frac{1}{k}\sum_{i\in N_k(x)}y_i,
\qquad
\hat f_k(x)=\mathbb 1\{\hat p_k(x)\geq 1/2\}.$$

$\hat p_k(x)$ é a proporção local da classe 1, e $\mathbb 1\{\cdot\}$
vale 1 quando a condição é verdadeira.

## 4. Por que padronizar?

In [ ]:
demo = df[["hemo", "bgr", "wc"]].dropna()
demo.std().round(2)

> **Interpretação**
>
> As escalas originais são muito diferentes. Sem padronização, uma
> diferença em `wc` poderia dominar a distância mesmo quando variações
> em hemoglobina ou glicemia fossem mais informativas. Padronizar não
> escolhe a importância clínica dos atributos; apenas evita que a
> unidade de medida determine o resultado.

A padronização $z_{ij}=(x_{ij}-\bar x_j)/s_j$ evita que a unidade
numericamente maior domine a distância. Média, desvio-padrão e
imputações devem ser estimados sem consultar o teste.

## 5. Separação e baseline

In [ ]:
X = df[numeric + categorical]
y = (df["classification"] == "ckd").astype(int)

X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=20260827
)

print("desenvolvimento:", X_dev.shape, "teste:", X_test.shape)
print("baseline majoritário:", round(y_dev.value_counts(normalize=True).max(), 3))

> **Interpretação**
>
> O desenvolvimento serve para aprender transformações e escolher $k$. O
> teste fica reservado até o procedimento estar pronto. O baseline
> majoritário mede o que obteríamos sem usar os atributos.

## 6. Pipeline sem vazamento

In [ ]:
preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), numeric),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical),
])

def knn_pipeline(k):
    return Pipeline([
        ("prep", preprocess),
        ("knn", KNeighborsClassifier(n_neighbors=k)),
    ])

A pipeline reaprende imputação, escala e codificação dentro de cada
fold. Fazer essas transformações antes da validação revelaria informação
dos folds de validação ao treinamento.

## 7. Escolha de $k$ por validação cruzada

In [ ]:
folds = StratifiedKFold(5, shuffle=True, random_state=7)
rows = []
for k in range(1, 32, 2):
    scores = cross_validate(
        knn_pipeline(k), X_dev, y_dev, cv=folds,
        scoring={"accuracy": "accuracy", "precision": "precision", "recall": "recall"}
    )
    rows.append({
        "k": k,
        "acurácia": scores["test_accuracy"].mean(),
        "precisão": scores["test_precision"].mean(),
        "recall": scores["test_recall"].mean(),
    })

cv = pd.DataFrame(rows)
best_k = int(cv.loc[cv["acurácia"].idxmax(), "k"])
cv.round(3)

In [ ]:
ax = cv.plot(x="k", y=["acurácia", "precisão", "recall"], marker="o", figsize=(9, 4.5))
ax.axvline(best_k, color="black", linestyle="--", label=f"k={best_k}")
ax.set(ylabel="média na validação", ylim=(0.75, 1.01), title="Escolha de k no desenvolvimento")
plt.show()

> **Interpretação**
>
> $k$ pequeno produz um modelo flexível; $k$ grande suaviza a fronteira.
> Nesta divisão, $k=1$ maximiza a acurácia média, mas uma escolha
> orientada por recall poderia seguir outro critério. A métrica deve ser
> definida pelo problema, não escolhida depois de observar o resultado
> mais favorável.

## 8. Avaliação final

In [ ]:
final = knn_pipeline(best_k)
final.fit(X_dev, y_dev)
pred = final.predict(X_test)

metrics = pd.Series({
    "acurácia": accuracy_score(y_test, pred),
    "precisão": precision_score(y_test, pred),
    "recall": recall_score(y_test, pred),
    "F1": f1_score(y_test, pred),
})
metrics.round(3)

As métricas são definidas a partir de verdadeiros positivos (VP),
verdadeiros negativos (VN), falsos positivos (FP) e falsos negativos
(FN):

$$\operatorname{Acurácia}=\frac{VP+VN}{VP+VN+FP+FN},\quad
\operatorname{Precisão}=\frac{VP}{VP+FP},\quad
\operatorname{Recall}=\frac{VP}{VP+FN}.$$

In [ ]:
cm = confusion_matrix(y_test, pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["sem DRC", "com DRC"],
            yticklabels=["sem DRC", "com DRC"])
plt.xlabel("previsto")
plt.ylabel("verdadeiro")
plt.title("Matriz de confusão no teste reservado")
plt.show()

> **Interpretação**
>
> O modelo acerta 79 de 80 casos, mas o erro é um falso negativo. Em
> triagem, esse tipo de erro pode ter custo especialmente alto. A
> acurácia isolada não mostra isso; recall e matriz de confusão mostram.

## 9. Inspeção do erro

In [ ]:
erros = X_test.copy()
erros["verdadeiro"] = y_test
erros["previsto"] = pred
erros.loc[erros["verdadeiro"] != erros["previsto"],
           ["age", "bp", "bgr", "sc", "hemo", "htn", "dm", "verdadeiro", "previsto"]]

> **Interpretação**
>
> Um erro individual não prova uma causa, mas ajuda a formular
> perguntas: o perfil está em uma região de sobreposição? Há muitas
> ausências? Esse subgrupo foi pouco representado? A análise de erros
> deve orientar investigação, não justificativas pós-hoc.

## 10. Extensões e limites

No KNN ponderado, vizinhos próximos recebem pesos maiores:

$$\hat p(x)=\frac{\sum_{i\in N_k(x)}w_i(x)y_i}{\sum_{i\in N_k(x)}w_i(x)},
\qquad w_i(x)=\frac{1}{d(x,x_i)+\varepsilon}.$$

Para regressão, substituímos o voto pela média local:
$\hat f_k(x)=k^{-1}\sum_{i\in N_k(x)}y_i$.

Limitações centrais: previsão custosa, sensibilidade à representação,
degradação em alta dimensão e ausência de extrapolação para regiões sem
exemplos.

## 11. KNN passo a passo

1.  **Definir a resposta e os atributos.** O que significa ser classe
    positiva e quais informações existem para uma nova pessoa?
2.  **Separar desenvolvimento e teste.** Todas as decisões seguintes
    usam apenas o desenvolvimento.
3.  **Tratar valores ausentes e categorias.** Essas transformações são
    aprendidas dentro da pipeline.
4.  **Colocar atributos em escalas comparáveis.** KNN usa distância; uma
    variável com números maiores pode dominar sem padronização.
5.  **Calcular as distâncias.** Para cada nova observação, medimos sua
    proximidade aos exemplos de treinamento.
6.  **Selecionar os $k$ mais próximos.** A previsão de classificação usa
    o voto; na regressão, usa a média das respostas vizinhas.
7.  **Escolher $k$ por validação.** $k$ pequeno reage a detalhes e
    ruído; $k$ grande produz decisões mais suaves.
8.  **Avaliar no teste reservado.** Matriz de confusão e métricas
    revelam tipos de acerto e erro.

## Exemplo curto de uma previsão

Considere cinco vizinhos com classes `1, 1, 0, 1, 0`. Para $k=5$, três
dos cinco são positivos, então a frequência local é $3/5=0{,}60$ e a
previsão, com limiar 0,5, é classe 1.

Essa frequência não é automaticamente uma probabilidade bem calibrada.
Ela diz como as classes estão distribuídas naquela vizinhança da
amostra.

## Por que muitas dimensões dificultam o método?

Com poucos atributos, é fácil imaginar pontos realmente próximos. Ao
adicionar muitas dimensões, os dados ficam espalhados e mesmo o vizinho
mais próximo pode estar distante em várias coordenadas. O método passa a
precisar de mais dados e fica mais sensível à escolha dos atributos.

Por isso, antes de aplicar KNN em uma base grande, vale perguntar se
todas as variáveis ajudam a definir semelhança. Remover ruído ou
construir uma representação melhor pode ser mais importante do que
testar muitos valores de $k$.

## Como ler as métricas

- acurácia: quantos casos foram classificados corretamente no total;
- precisão: entre os previstos como positivos, quantos eram positivos;
- recall: entre os positivos reais, quantos foram encontrados;
- F1: compromisso entre precisão e recall.

Na base renal, um único falso negativo merece atenção mesmo quando a
acurácia é alta. A métrica deve refletir a consequência do erro, e não
apenas produzir o maior número.

> **Escopo do resultado**
>
> O teste avalia esta pipeline nesta amostra. Ele não transforma o
> exercício em ferramenta clínica nem garante que o desempenho se repita
> em outro hospital.

## 12. Checklist final

- pergunta, população, unidade e momento da previsão estão definidos?
- o teste permaneceu isolado?
- pré-processamento está dentro da pipeline?
- há um baseline?
- a métrica representa o custo dos erros?
- foram mostradas saídas, matriz de confusão e exemplos de erro?
- limitações e escopo de uso estão documentados?

## Referências

- James et al., *An Introduction to Statistical Learning*, capítulos 2,
  4 e 5.
- Géron, *Hands-On Machine Learning*, capítulos 2 e 3.
- Kuhn e Johnson, *Applied Predictive Modeling*, capítulos 4, 7 e 13.
- [Chronic Kidney Disease Dataset —
  Kaggle](https://www.kaggle.com/datasets/mansoordaku/ckdisease).